In [11]:
import torch
import torchvision.models as models
from torch import nn
import torch.nn.functional as F
from typing import Dict, Union, List

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Model initialization

In [12]:
# Initialize ResNet-50 model
resnet50 = models.resnet50(pretrained=True)

# Move model to device
resnet50 = resnet50.to(device)

# Display model info
print("ResNet-50 Model loaded successfully")
print(f"Model is on device: {next(resnet50.parameters()).device}")

# Weight map - maps class/label to a given object weight

In [13]:
# Class Weight Hashmap
# Maps class labels to their corresponding weights
class_weight_map: Dict[str, float] = {
    'cat': 1.2,
    'dog': 1.5,
    'bird': 0.8,
    'car': 1.0,
    'person': 1.3,
    'tree': 0.9,
    'goldfish': 99.99
}

print("Initial Class Weight Mapping:")
for class_label, weight in class_weight_map.items():
    print(f"  {class_label}: {weight}")

# Getting or setting the weight of an object in the hashmap

In [14]:
def get_class_weight(class_label: str, default_weight: float = 1.0) -> float:
    """
    Infer the weight for a given class label.
    
    Args:
        class_label: The class label to look up
        default_weight: The default weight for unknown classes (default: 1.0)
        
    Returns:
        The weight associated with the class label, or default_weight if not found
    """
    return class_weight_map.get(class_label.lower(), default_weight)


def get_class_weights_batch(class_labels: List[str], default_weight: float = 1.0) -> torch.Tensor:
    """
    Get weights for a batch of class labels.
    
    Args:
        class_labels: List of class labels
        default_weight: The default weight for unknown classes
        
    Returns:
        A tensor of weights corresponding to the input labels
    """
    weights = [get_class_weight(label, default_weight) for label in class_labels]
    return torch.tensor(weights, dtype=torch.float32)


def add_class_weight(class_label: str, weight: float) -> None:
    """
    Add or update a class weight in the hashmap.
    
    Args:
        class_label: The class label
        weight: The weight value for this class
    """
    class_weight_map[class_label.lower()] = weight
    print(f"Added/Updated: {class_label} -> {weight}")


# Test the weight inference
print("\n--- Testing Weight Inference ---")
print(f"Weight for 'dog': {get_class_weight('dog')}")
print(f"Weight for 'DOG' (case insensitive): {get_class_weight('DOG')}")
print(f"Weight for 'unknown_class': {get_class_weight('unknown_class')}")

print("\n--- Testing Batch Weight Inference ---")
test_batch = ['cat', 'dog', 'unknown', 'tree']
batch_weights = get_class_weights_batch(test_batch)
print(f"Batch labels: {test_batch}")
print(f"Batch weights: {batch_weights}")

# Preprocess images

In [16]:
from PIL import Image
import torchvision.transforms as transforms
import urllib.request

# Image preprocessing pipeline - matches ImageNet normalization
image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Download ImageNet class labels
try:
    with urllib.request.urlopen('https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt') as url:
        imagenet_classes = [line.decode('utf-8').strip() for line in url.readlines()]
    print(f"✓ Loaded {len(imagenet_classes)} ImageNet classes")
except Exception as e:
    print(f"Warning: Could not download ImageNet classes: {e}")
    imagenet_classes = None

# Test inference function

In [17]:
def test_inference(image_path: str):
    """Test ResNet-50 inference on an image and return predictions with weights"""
    try:
        image = Image.open(image_path).convert('RGB')
        image_tensor = image_transforms(image).unsqueeze(0).to(device)
        
        resnet50.eval()
        with torch.no_grad():
            outputs = resnet50(image_tensor)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            top_prob, top_indices = torch.topk(probabilities, k=5)
        
        results = []
        for i, (prob, idx) in enumerate(zip(top_prob[0], top_indices[0])):
            class_idx = idx.item()
            confidence = prob.item()
            class_name = imagenet_classes[class_idx].split(',')[0] if imagenet_classes else f"class_{class_idx}"
            weight = get_class_weight(class_name)
            
            results.append({
                'rank': i + 1,
                'class': class_name,
                'confidence': f"{confidence:.4f}",
                'weight': weight
            })
        
        return results
    
    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}")
        return None
    except Exception as e:
        print(f"Error during inference: {e}")
        return None
    
    


# Example usage:
print("\nTo test inference on an image, use:")
print("  results = test_inference('path/to/your/image.jpg')")
print("  for r in results:")
print("      print(f\"{r['rank']}. {r['class']} ({r['confidence']}) - Weight: {r['weight']}\")")

# Uplaod image from URL and test inference

In [18]:
def test_inference_from_url(url: str):
    """Test inference on image from URL"""
    import urllib.request
    
    # Add User-Agent header to avoid 403 Forbidden errors
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    req = urllib.request.Request(url, headers=headers)
    
    try:
        with urllib.request.urlopen(req) as response:
            with open('temp_image.jpg', 'wb') as out_file:
                out_file.write(response.read())
        return test_inference('temp_image.jpg')
    except Exception as e:
        print(f"Error downloading image: {e}")
        return None

test_inference_from_url('https://github.com/EliSchwartz/imagenet-sample-images/blob/master/n01443537_goldfish.JPEG?raw=true')
# upload_image_colab()  # Uncomment to use in Colab

# Usage:
# upload_image_colab()  # Interactive upload
# OR
# test_inference_from_url('https://example.com/image.jpg')